# Lesson 2A: Hierarchical Clustering Theory

## Introduction

K-Means forces you to choose K upfront. But what if you don't know how many clusters exist?

**Hierarchical clustering** builds a tree of nested clusters, revealing structure at all scales. This **dendrogram** shows:
- At the bottom: each point is its own cluster
- As you move up: similar clusters merge together
- At the top: all points in one cluster

You cut the dendrogram at any height to get K clusters — no K specified upfront!

In this lesson, we will:
1. Understand agglomerative vs. divisive hierarchical clustering
2. Learn linkage methods: single, complete, average, ward
3. Build hierarchical clustering from scratch (compute linkage matrix)
4. Visualize and interpret dendrograms
5. Compare linkage methods on real data
6. Cross-validate with scikit-learn

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Hierarchical Clustering Paradigms](#paradigms)
4. [Agglomerative Clustering Algorithm](#agglomerative)
5. [Linkage Methods](#linkage)
6. [From-Scratch Implementation](#scratch)
7. [Dendrograms](#dendrograms)
8. [Comparing Linkage Methods](#comparing)
9. [Scikit-learn Validation](#sklearn)
10. [Cutting Dendrograms](#cutting)
11. [Advantages and Disadvantages](#pros-cons)
12. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import pdist, squareform, euclidean, cdist
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score
from typing import Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="paradigms"></a>
## Hierarchical Clustering Paradigms

### Agglomerative (Bottom-Up)

Start with each point as its own cluster. Repeatedly merge the two most similar clusters until all points are in one cluster.

**Pseudocode**:
```
1. Initialize: Each point is a cluster (n clusters)
2. Repeat until 1 cluster remains:
   a. Find two most similar clusters
   b. Merge them
   c. Record merge (for dendrogram)
3. Return: Dendrogram (merge history)
```

### Divisive (Top-Down)

Start with all points in one cluster. Recursively split clusters until each point is its own cluster.

**Less common** — more computationally expensive. We focus on agglomerative.

<a name="agglomerative"></a>
## Agglomerative Clustering Algorithm

### Key Idea: Linkage

When merging two clusters, how do we decide which clusters to merge next? We need a **linkage criterion** that measures distance between clusters.

Different linkage methods create different dendrograms from the same data!

<a name="linkage"></a>
## Linkage Methods

Given two clusters $A$ and $B$, distance $d(A, B)$ is computed as:

### 1. Single Linkage (Nearest Neighbor)

$$d(A, B) = \min_{\mathbf{a} \in A, \mathbf{b} \in B} \|\mathbf{a} - \mathbf{b}\|$$

Distance between clusters = distance between their closest points.

**Effect**: Tends to create **long, thin chains** (chaining problem).

### 2. Complete Linkage (Farthest Neighbor)

$$d(A, B) = \max_{\mathbf{a} \in A, \mathbf{b} \in B} \|\mathbf{a} - \mathbf{b}\|$$

Distance between clusters = distance between their farthest points.

**Effect**: Creates **more compact, balanced clusters**.

### 3. Average Linkage

$$d(A, B) = \frac{1}{|A| |B|} \sum_{\mathbf{a} \in A} \sum_{\mathbf{b} \in B} \|\mathbf{a} - \mathbf{b}\|$$

Distance between clusters = average distance between all pairs.

**Effect**: Compromise between single and complete.

### 4. Ward Linkage

Minimize the increase in within-cluster sum of squares when merging.

$$d(A, B) = \sqrt{\frac{|A| |B|}{|A| + |B|}} \cdot \|\boldsymbol{\mu}_A - \boldsymbol{\mu}_B\|$$

Where $\boldsymbol{\mu}_A$, $\boldsymbol{\mu}_B$ are cluster means.

**Effect**: **Most similar to K-Means behavior** — favors compact, roughly equal-sized clusters.

In [ ]:
# Visualize linkage concepts
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Create two clusters
np.random.seed(42)
A = np.random.normal([0, 0], 0.3, (5, 2))
B = np.random.normal([3, 3], 0.3, (5, 2))

ax = axes[0, 0]
ax.scatter(A[:, 0], A[:, 1], c='red', s=100, label='Cluster A', alpha=0.6)
ax.scatter(B[:, 0], B[:, 1], c='blue', s=100, label='Cluster B', alpha=0.6)

# Single linkage: nearest
dists_single = cdist(A, B)
min_idx = np.unravel_index(np.argmin(dists_single), dists_single.shape)
ax.plot([A[min_idx[0], 0], B[min_idx[1], 0]], [A[min_idx[0], 1], B[min_idx[1], 1]],
        'g-', linewidth=3, label=f'Nearest: {dists_single[min_idx[0], min_idx[1]]:.2f}')
ax.set_title('Single Linkage (Nearest)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Complete linkage: farthest
ax = axes[0, 1]
ax.scatter(A[:, 0], A[:, 1], c='red', s=100, label='Cluster A', alpha=0.6)
ax.scatter(B[:, 0], B[:, 1], c='blue', s=100, label='Cluster B', alpha=0.6)
max_idx = np.unravel_index(np.argmax(dists_single), dists_single.shape)
ax.plot([A[max_idx[0], 0], B[max_idx[1], 0]], [A[max_idx[0], 1], B[max_idx[1], 1]],
        'g-', linewidth=3, label=f'Farthest: {dists_single[max_idx[0], max_idx[1]]:.2f}')
ax.set_title('Complete Linkage (Farthest)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Average linkage: all pairs
ax = axes[1, 0]
ax.scatter(A[:, 0], A[:, 1], c='red', s=100, label='Cluster A', alpha=0.6)
ax.scatter(B[:, 0], B[:, 1], c='blue', s=100, label='Cluster B', alpha=0.6)
avg_dist = dists_single.mean()
ax.scatter([A.mean(0)[0], B.mean(0)[0]], [A.mean(0)[1], B.mean(0)[1]],
          c='green', s=200, marker='*', label=f'Centroids', zorder=5)
ax.set_title(f'Average Linkage: {avg_dist:.2f}', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Ward linkage: minimize merge cost
ax = axes[1, 1]
ax.scatter(A[:, 0], A[:, 1], c='red', s=100, label='Cluster A', alpha=0.6)
ax.scatter(B[:, 0], B[:, 1], c='blue', s=100, label='Cluster B', alpha=0.6)
centroid_A = A.mean(0)
centroid_B = B.mean(0)
ax.plot([centroid_A[0], centroid_B[0]], [centroid_A[1], centroid_B[1]],
        'g-', linewidth=3, label=f'Centroids distance')
ax.scatter([centroid_A[0], centroid_B[0]], [centroid_A[1], centroid_B[1]],
          c='green', s=200, marker='*', zorder=5)
ax.set_title('Ward Linkage (Minimize WCSS increase)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("💡 Different linkage methods measure inter-cluster distance differently")

<a name="scratch"></a>
## From-Scratch Hierarchical Clustering

Let's implement agglomerative clustering with single linkage (simplest).

In [ ]:
def hierarchical_clustering_scratch(X: NDArray, linkage_method: str = 'single') -> Tuple[NDArray, list]:
    """
    Agglomerative hierarchical clustering from scratch.
    
    Returns:
        Z: Linkage matrix (scipy.cluster.hierarchy format)
        clusters: Cluster assignments at each merge step
    """
    n_samples = X.shape[0]
    
    # Initialize: each point is its own cluster
    clusters = [[i] for i in range(n_samples)]
    
    # Pairwise distances
    from scipy.spatial.distance import pdist, squareform
    pairwise_dists = squareform(pdist(X, metric='euclidean'))
    
    # Store linkage history
    Z = []
    
    # Merge until one cluster remains
    cluster_id_counter = n_samples
    
    while len(clusters) > 1:
        # Find closest pair of clusters
        min_dist = float('inf')
        merge_i, merge_j = 0, 1
        
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                # Compute distance based on linkage method
                if linkage_method == 'single':
                    dist = np.min([pairwise_dists[a, b] for a in clusters[i] for b in clusters[j]])
                elif linkage_method == 'complete':
                    dist = np.max([pairwise_dists[a, b] for a in clusters[i] for b in clusters[j]])
                elif linkage_method == 'average':
                    dist = np.mean([pairwise_dists[a, b] for a in clusters[i] for b in clusters[j]])
                
                if dist < min_dist:
                    min_dist = dist
                    merge_i, merge_j = i, j
        
        # Record merge in linkage matrix format
        # [cluster_i_id, cluster_j_id, distance, sample_count]
        Z.append([merge_i, merge_j, min_dist, len(clusters[merge_i]) + len(clusters[merge_j])])
        
        # Merge clusters
        clusters[merge_i] = clusters[merge_i] + clusters[merge_j]
        clusters.pop(merge_j)
    
    return np.array(Z), clusters

# Test on small synthetic data
X_small, _ = make_blobs(n_samples=10, centers=3, n_features=2, random_state=42)

Z, clusters = hierarchical_clustering_scratch(X_small, linkage_method='single')

print(f"Linkage matrix shape: {Z.shape}")
print(f"First few merges:")
print(Z[:5])

<a name="dendrograms"></a>
## Dendrograms: Visualizing Hierarchical Structure

In [ ]:
# Use scipy's efficient implementation
from scipy.cluster.hierarchy import linkage, dendrogram

X, y_true = make_blobs(n_samples=30, centers=3, n_features=2, random_state=42)

# Compute linkage matrices for different methods
Z_single = linkage(X, method='single')
Z_complete = linkage(X, method='complete')
Z_average = linkage(X, method='average')
Z_ward = linkage(X, method='ward')

# Plot dendrograms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

linkage_methods = [('single', Z_single), ('complete', Z_complete), ('average', Z_average), ('ward', Z_ward)]

for idx, (method, Z) in enumerate(linkage_methods):
    ax = axes[idx // 2, idx % 2]
    dendrogram(Z, ax=ax)
    ax.set_title(f'{method.capitalize()} Linkage', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Index')
    ax.set_ylabel('Distance')

plt.tight_layout()
plt.show()

print("💡 Single linkage shows chaining (long thin structure)")
print("💡 Complete linkage shows more balanced clusters")
print("💡 Ward resembles K-Means-like structure")

<a name="sklearn"></a>
## Scikit-learn Validation

In [ ]:
# Use AgglomerativeClustering from scikit-learn
# Cut dendrograms at height that creates 3 clusters

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (method, Z) in enumerate(linkage_methods):
    # Get cluster assignments at 3 clusters
    from scipy.cluster.hierarchy import fcluster
    labels = fcluster(Z, t=3, criterion='maxclust') - 1  # Convert to 0-indexed
    
    # Verify with sklearn
    agg = AgglomerativeClustering(n_clusters=3, linkage=method)
    labels_sklearn = agg.fit_predict(X)
    
    # Plot
    ax = axes[idx // 2, idx % 2]
    scatter = ax.scatter(X[:, 0], X[:, 1], c=labels_sklearn, cmap='viridis', s=100, alpha=0.6, edgecolors='k')
    ax.set_title(f'{method.capitalize()} Linkage (K=3)', fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')
    ax.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax)

plt.tight_layout()
plt.show()

print("✅ Scipy and scikit-learn produce consistent results!")

<a name="cutting"></a>
## Cutting Dendrograms to Get K Clusters

The key advantage of hierarchical clustering: **choose K by cutting the dendrogram**.

In [ ]:
# Iris dataset
iris = load_iris()
X_iris = iris.data
X_iris = StandardScaler().fit_transform(X_iris)

# Compute linkage
Z_iris = linkage(X_iris, method='ward')

# Plot dendrogram with cut lines for different K
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Full dendrogram
ax = axes[0]
dendrogram(Z_iris, ax=ax, leaf_rotation=90, leaf_font_size=8)
ax.set_title('Iris Dendrogram (Ward Linkage)', fontsize=12, fontweight='bold')
ax.set_ylabel('Distance')

# Add cut lines for different K values
cut_heights = [15, 20, 25]
colors = ['red', 'green', 'blue']
for h, color in zip(cut_heights, colors):
    ax.axhline(y=h, color=color, linestyle='--', alpha=0.7, label=f'K={int(Z_iris[-10, 2]) - int(h)}')
ax.legend()

# Compare cluster assignments at different K
ax = axes[1]
for k in [2, 3, 4]:
    labels = fcluster(Z_iris, t=k, criterion='maxclust')
    print(f"K={k}: {len(np.unique(labels))} clusters")
    print(f"  Silhouette: {silhouette_score(X_iris, labels):.3f}")
    print(f"  Davies-Bouldin: {davies_bouldin_score(X_iris, labels):.3f}")

ax.text(0.1, 0.9, "K-selection metrics:\n(higher silhouette → better)\n(lower DB → better)",
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax.axis('off')

plt.tight_layout()
plt.show()

<a name="pros-cons"></a>
## Advantages and Disadvantages

### ✅ Advantages

1. **No K required upfront** — dendrograms show structure at all scales
2. **Interpretable** — tree structure reveals how clusters relate
3. **Flexible** — try different linkage methods
4. **Handles arbitrary shapes** — unlike K-Means
5. **Dendrogram visualization** is intuitive

### ⚠️ Disadvantages

1. **Computationally expensive** — O(n²) or O(n² log n) time and memory
2. **Sensitive to linkage choice** — different methods give different results
3. **Greedy algorithm** — no global optimization (merges are never undone)
4. **Chaining problem** — single linkage can create spurious chains
5. **Not scalable** — for n > 100k, use approximations or other methods

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Hierarchical clustering** builds a tree of nested clusters without specifying K upfront

2. **Agglomerative approach**: Start with points as clusters, merge upward (most common)

3. **Linkage methods**:
   - Single: Nearest neighbor (prone to chaining)
   - Complete: Farthest neighbor (compact clusters)
   - Average: Compromise
   - Ward: Minimize WCSS (similar to K-Means)

4. **Dendrograms** show structure clearly; cut at appropriate height to get K clusters

5. **Trade-off**: More interpretable than K-Means but less scalable

### Next Steps

In Lesson 2B (practical), we will:
- Cut dendrograms programmatically
- Apply hierarchical clustering to real data
- Compare dendrograms from different linkage methods
- Benchmark scalability
- Combine with metrics from Lesson 0B to choose K

Then we move to **DBSCAN (Lesson 3)** for density-based clustering, which handles arbitrary cluster shapes and noise.